# Обучкение модели RandomForest для задачи бинарной классификации - предсказания купит ли покупатель товар

In [ ]:
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

In [ ]:
SEED = 42
TEST_SIZE = 0.20

DATA_PATH = "online_shoppers_intention.csv"
ARTIFACT_PATH = "artifact/model.joblib"

ARTIFACT_VERSION = "1.0.0"

CLASSIFICATION_THRESHOLD = 0.35

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_leaf": 5,
    "random_state": SEED,
    "n_jobs": -1,
}

## 1. Загрузка данных

In [ ]:
df = pd.read_csv(DATA_PATH)

TARGET = "Revenue"

df = df.drop_duplicates().reset_index(drop=True)

df[TARGET] = df[TARGET].astype(int)
df["Weekend"] = df["Weekend"].astype(int)

CODE_CAT_COLS = [
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
]
df[CODE_CAT_COLS] = df[CODE_CAT_COLS].astype(str)

X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print(f"Всего объектов после удаления дубликатов: {len(df)}")
print(f"Train: {X_train.shape}")
print(f"Test:  {X_test.shape}")
print(f"Доля класса 1: train={y_train.mean():.3f}, test={y_test.mean():.3f}")

## 2. Feature engineering

In [ ]:
# class FeatureEngineering(BaseEstimator, TransformerMixin):
#
#     def fit(self, X, y=None):
#         return self
#
#     @staticmethod
#     def _ratio(duration, count):
#         count = np.asarray(count, dtype=float)
#         duration = np.asarray(duration, dtype=float)
#
#         return np.divide(
#             duration,
#             count,
#             out=np.zeros_like(duration, dtype=float),
#             where=count != 0,
#         )
#
#     def transform(self, X):
#         X = X.copy()
#
#         # Замена Duration на Duration / Count
#         X['Administrative_Duration_per_page'] = self._ratio(
#             X['Administrative_Duration'],
#             X['Administrative'],
#         )
#         X['Informational_Duration_per_page'] = self._ratio(
#             X['Informational_Duration'],
#             X['Informational'],
#         )
#         X['ProductRelated_Duration_per_page'] = self._ratio(
#             X['ProductRelated_Duration'],
#             X['ProductRelated'],
#         )
#
#         # Бинаризация (X > 0)
#         X['has_BounceRates'] = (X['BounceRates'] > 0).astype(int)
#         X['has_PageValues'] = (X['PageValues'] > 0).astype(int)
#         X['has_Informational'] = (X['Informational'] > 0).astype(int)
#         X['has_SpecialDay'] = (X['SpecialDay'] > 0).astype(int)
#
#
#         X = X.drop(
#             columns=[
#                 'Administrative_Duration',
#                 'Informational_Duration',
#                 'ProductRelated_Duration',
#                 'BounceRates',
#                 'PageValues',
#                 'Informational',
#                 'SpecialDay',
#             ]
#         )
#
#         return X

## 3. Предобработка

In [ ]:
LOG_COLS = [
    "Administrative",
    "Administrative_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "PageValues",
    "Informational",
    "Informational_Duration",
]

NUM_COLS = [
    "BounceRates",
    "ExitRates",
    "SpecialDay",
]

CAT_COLS = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
]

BIN_COLS = ["Weekend"]


def make_preprocessor():
    log_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ]
    )

    num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

    categorical_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "ohe",
                OneHotEncoder(
                    handle_unknown="infrequent_if_exist",
                    min_frequency=0.01,
                    sparse_output=False,
                ),
            ),
        ]
    )

    binary_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent"))])

    return ColumnTransformer(
        transformers=[
            ("log_num", log_pipeline, LOG_COLS),
            ("num", num_pipeline, NUM_COLS),
            ("cat", categorical_pipeline, CAT_COLS),
            ("bin", binary_pipeline, BIN_COLS),
        ],
        remainder="drop",
    )

## 4. Pipeline и RandomForest

In [ ]:
pipeline = Pipeline(
    [
        ("preprocessing", make_preprocessor()),
        ("model", RandomForestClassifier(**RF_PARAMS)),
    ]
)

pipeline

## 5. Обучение

In [ ]:
pipeline.fit(X_train, y_train)

## 6. Финальная проверка на тестовой выборке

In [ ]:
test_proba = pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= CLASSIFICATION_THRESHOLD).astype(int)

test_metrics = {
    "average_precision": average_precision_score(y_test, test_proba),
    "roc_auc": roc_auc_score(y_test, test_proba),
    "f1": f1_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred),
    "recall": recall_score(y_test, test_pred),
    "accuracy": accuracy_score(y_test, test_pred),
}

pd.Series(test_metrics).round(4)

In [ ]:
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["нет покупки", "покупка"],
        digits=4,
    )
)

In [ ]:
cm = confusion_matrix(y_test, test_pred)

print("Матрица ошибок:")
cm

## 7. Metadata

In [ ]:
FEATURES = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay",
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

assert FEATURES == list(X_train.columns), (
    "Порядок входных признаков изменился. " "Проверьте схему исходных данных."
)

metadata = {
    "model_version": ARTIFACT_VERSION,
    "features": FEATURES,
    "threshold": CLASSIFICATION_THRESHOLD,
}

metadata

## 8. Сохранение joblib-бандла

In [ ]:
bundle = {
    "pipeline": pipeline,
    "metadata": metadata,
}

joblib.dump(bundle, ARTIFACT_PATH)

print(f"Артефакт сохранен: {ARTIFACT_PATH}")